# Web Crawler Demo: CNN

This notebook demonstrates the WebCrawler Spider by crawling CNN.com and analyzing the results.

In [8]:
import json
import logging
from collections import Counter

import pandas as pd

from WebCrawler import Serializers, Spider

# Configure logging to see crawler activity
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)

## Set up and run the Spider

We'll crawl CNN with a depth of 3 to explore multiple levels of the site. **Note:** Each cell's cached output reflects when it was last run. Re-execute cells to see fresh results.

In [10]:
# Create Spider instance
start_url = "https://www.cnn.com"
max_depth = 3  # Keep it shallow to avoid excessive requests

spider = Spider(start_url=start_url, max_depth=max_depth, debug=True)

print(f"Starting crawl of {start_url} (max depth: {max_depth})...\n")

# Run the async crawler (await works in Jupyter)
documents = await spider.run_async()

print(f"\nCrawl complete! Visited {len(documents)} pages.")

2026-06-08 14:59:56,127 WebCrawler.Spider DEBUG    Spider initialized: strategy=BFS, max_depth=3
2026-06-08 14:59:56,127 WebCrawler.Spider DEBUG    Spider initialized: strategy=BFS, max_depth=3
2026-06-08 14:59:56,127 - WebCrawler.Spider - DEBUG - Spider initialized: strategy=BFS, max_depth=3
2026-06-08 14:59:56,180 WebCrawler.Spider DEBUG    Fetching https://www.cnn.com (attempt 1)
2026-06-08 14:59:56,180 WebCrawler.Spider DEBUG    Fetching https://www.cnn.com (attempt 1)
2026-06-08 14:59:56,180 WebCrawler.Spider DEBUG    Fetching https://www.cnn.com (attempt 1)
2026-06-08 14:59:56,180 - WebCrawler.Spider - DEBUG - Fetching https://www.cnn.com (attempt 1)
2026-06-08 14:59:56,249 WebCrawler.Spider INFO     Visited: https://www.cnn.com (Total Visited: 1)
2026-06-08 14:59:56,249 WebCrawler.Spider INFO     Visited: https://www.cnn.com (Total Visited: 1)
2026-06-08 14:59:56,249 WebCrawler.Spider INFO     Visited: https://www.cnn.com (Total Visited: 1)
2026-06-08 14:59:56,249 - WebCrawler.S

Starting crawl of https://www.cnn.com (max depth: 3)...


Crawl complete! Visited 1 pages.


## Analyze Results

In [11]:
# Display page titles and link counts
print(f"{'URL':<60} {'Title':<40} {'Links':<8}")
print("-" * 110)

for doc in documents:
    url_short = doc.url[:59] if len(doc.url) > 59 else doc.url
    title_short = doc.title[:39] if len(doc.title) > 39 else doc.title
    num_links = len(doc.links)
    print(f"{url_short:<60} {title_short:<40} {num_links:<8}")

URL                                                          Title                                    Links   
--------------------------------------------------------------------------------------------------------------
https://www.cnn.com                                          Breaking News, Latest News and Videos |  232     


## Statistics

In [12]:
# Aggregate statistics
total_links = sum(len(doc.links) for doc in documents)
total_internal = sum(len(doc.internal_links) for doc in documents)
total_external = sum(len(doc.external_links) for doc in documents)

print(f"Total pages crawled: {len(documents)}")
print(f"Total links found: {total_links}")
print(f"  - Internal: {total_internal}")
print(f"  - External: {total_external}")
print(f"\nAverage links per page: {total_links / len(documents):.1f}")

Total pages crawled: 1
Total links found: 232
  - Internal: 209
  - External: 23

Average links per page: 232.0


## Sample External Links from Homepage

In [13]:
# Show external links from the first page (homepage)
if documents:
    homepage = documents[0]
    print(f"External links from {homepage.url}:\n")
    for link in homepage.external_links[:10]:  # Show first 10
        print(f"  • {link.url}")
    if len(homepage.external_links) > 10:
        print(f"  ... and {len(homepage.external_links) - 10} more")

External links from https://www.cnn.com:

  • https://us.cnn.com?hpt=header_edition-picker
  • https://edition.cnn.com?hpt=header_edition-picker
  • https://arabic.cnn.com?hpt=header_edition-picker
  • https://cnnespanol.cnn.com/?hpt=header_edition-picker
  • https://bleacherreport.com/
  • https://www.cnn10.com
  • https://cnn.it/5thingsquiz
  • https://careers.wbd.com/cnnjobs
  • https://facebook.com/CNN
  • https://twitter.com/CNN
  ... and 13 more


## Sample Internal Links from Homepage

In [6]:
# Show internal links from the first page (homepage)
if documents:
    homepage = documents[0]
    print(f"Internal links from {homepage.url}:\n")
    for link in homepage.internal_links[:10]:  # Show first 10
        text_preview = (link.text[:40] + "...") if len(link.text) > 40 else link.text
        print(f"  • {link.url}")
        if text_preview:
            print(f"    → {text_preview}")
    if len(homepage.internal_links) > 10:
        print(f"  ... and {len(homepage.internal_links) - 10} more")

Internal links from https://www.cnn.com:

  • https://www.cnn.com/us
    → US
  • https://www.cnn.com/world
    → World
  • https://www.cnn.com/politics
    → Politics
  • https://www.cnn.com/business
    → Business
  • https://www.cnn.com/health
    → Health
  • https://www.cnn.com/entertainment
    → Entertainment
  • https://www.cnn.com/cnn-underscored
    → Underscored
  • https://www.cnn.com/style
    → Style
  • https://www.cnn.com/travel
    → Travel
  • https://www.cnn.com/sports
    → Sports
  ... and 197 more


## Domain Analysis

In [7]:
# Get domains from external links
external_domains = Counter()
for doc in documents:
    for link in doc.external_links:
        try:
            external_domains[link.url.split("/")[2]] += 1  # Extract domain from URL
        except Exception:
            pass

print("Most common external domains:")
for domain, count in external_domains.most_common(10):
    print(f"  {domain}: {count} links")

Most common external domains:
  bleacherreport.com: 7 links
  cnnespanol.cnn.com: 2 links
  cnn.onelink.me: 2 links
  us.cnn.com?hpt=header_edition-picker: 1 links
  edition.cnn.com?hpt=header_edition-picker: 1 links
  arabic.cnn.com?hpt=header_edition-picker: 1 links
  www.cnn10.com: 1 links
  cnn.it: 1 links
  careers.wbd.com: 1 links
  facebook.com: 1 links


## Export Data to Multiple Formats

The Serializers module allows exporting crawled documents to JSON, Pandas, Polars, or PyArrow formats. This is useful for data analysis and integration with other tools.

In [ ]:
# Create a serializer instance from the documents
serializer = Serializers(documents)

# Export to JSON
json_file = "cnn_crawl.json"
serializer.to_json(json_file, include_html=False)
print(f"✓ Exported to {json_file}")

# Read and show structure
with open(json_file) as f:
    data = json.load(f)

print("\nJSON structure (first page):")
print(f"  - URL: {data[0]['url']}")
print(f"  - Title: {data[0]['title']}")
print(f"  - Status: {data[0]['status_code']}")
print(f"  - Domain: {data[0]['domain']}")
print(f"  - Internal links: {len(data[0]['internal_links'])}")
print(f"  - External links: {len(data[0]['external_links'])}")

In [ ]:
# Export to Pandas DataFrame with flattened links
df = serializer.to_pandas()

print(f"Pandas DataFrame: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumns: {', '.join(df.columns)}")
print("\nFirst 5 links:")
print(df[["url", "title", "link_url", "link_type"]].head())

In [ ]:
# Analyze link types and domains
print("Link type distribution:")
print(df["link_type"].value_counts())

print("\n\nExternal link domains:")
# Extract domain from URL
df["external_domain"] = df[df["link_type"] == "external"]["link_url"].apply(
    lambda x: x.split("/")[2] if pd.notna(x) else None
)
print(df["external_domain"].value_counts().head(10))

In [ ]:
# Export to Polars (for comparison - often faster for large datasets)
df_polars = serializer.to_polars()

print(f"Polars DataFrame: {df_polars.shape[0]} rows × {df_polars.shape[1]} columns")
print("\nSchema:")
print(df_polars.schema)

In [ ]:
import pandas as pd


def extract_link_summary(doc):
    """Extract and return summary for each page."""
    return {
        "url": doc.url,
        "title": doc.title,
        "internal_links": len(doc.internal_links),
        "external_links": len(doc.external_links),
        "total_links": len(doc.links),
        "status": doc.status_code,
    }


# Run spider with result accumulation
spider_agg = Spider(
    start_url="https://www.cnn.com",
    max_depth=1,
    on_page_crawled=extract_link_summary,
    accumulate_results=True,  # Collect callback return values
)

print("Crawling with result accumulation...\n")
summaries = await spider_agg.run_async()

print(f"✓ Crawl complete! Collected {len(summaries)} page summaries\n")

# Display accumulated results
df_summaries = pd.DataFrame(summaries)

print("Aggregated Data (first 5 rows):")
print(df_summaries[["url", "title", "total_links"]].head())

print("\n\nSummary Statistics:")
print(f"  Total pages: {len(df_summaries)}")
print(f"  Avg links per page: {df_summaries['total_links'].mean():.1f}")
print(f"  Pages with errors: {(df_summaries['status'] != 200).sum()}")

In [ ]:
def extract_link_summary(doc):
    """Extract and return summary for each page."""
    return {
        "url": doc.url,
        "title": doc.title,
        "internal_links": len(doc.internal_links),
        "external_links": len(doc.external_links),
        "total_links": len(doc.links),
        "status": doc.status_code,
    }


# Run spider with result accumulation
spider_agg = Spider(
    start_url="https://www.cnn.com",
    max_depth=1,
    on_page_crawled=extract_link_summary,
    accumulate_results=True,  # Collect callback return values
)

print("Crawling with result accumulation...\n")
summaries = await spider_agg.run_async()

print(f"✓ Crawl complete! Collected {len(summaries)} page summaries\n")

# Display accumulated results
df_summaries = pd.DataFrame(summaries)

print("Aggregated Data (first 5 rows):")
print(df_summaries[["url", "title", "total_links"]].head())

print("\n\nSummary Statistics:")
print(f"  Total pages: {len(df_summaries)}")
print(f"  Avg links per page: {df_summaries['total_links'].mean():.1f}")
print(f"  Pages with errors: {(df_summaries['status'] != 200).sum()}")

## Aggregate Results with Callbacks

Use callbacks with `accumulate_results=True` to collect transformed data on-the-fly.

In [ ]:
import json

# Track some statistics
crawl_stats = {"total": 0, "errors": 0}


async def save_to_jsonl(doc):
    """Stream each page result to JSONL file immediately."""
    crawl_stats["total"] += 1
    with open("cnn_results.jsonl", "a") as f:
        json.dump(
            {
                "url": doc.url,
                "title": doc.title,
                "status": doc.status_code,
                "internal_links": len(doc.internal_links),
                "external_links": len(doc.external_links),
            },
            f,
        )
        f.write("\n")
    return doc.url  # Return for tracking


def on_error(url, exception):
    """Track errors during crawl."""
    crawl_stats["errors"] += 1
    print(f"❌ Failed to crawl {url}: {exception}")


# Clear previous results
import os

if os.path.exists("cnn_results.jsonl"):
    os.remove("cnn_results.jsonl")

# Run with callbacks - results stream to disk, not memory
spider_callback = Spider(
    start_url="https://www.cnn.com",
    max_depth=1,
    on_page_crawled=save_to_jsonl,
    on_error=on_error,
    accumulate_results=False,  # Don't keep in memory
)

print("Streaming results to cnn_results.jsonl (no memory buildup)...\n")
result = await spider_callback.run_async()

print("\n✓ Crawl complete!")
print(f"  Total pages: {crawl_stats['total']}")
print(f"  Errors: {crawl_stats['errors']}")
print(f"  Result list: {len(result)} (empty because accumulate_results=False)")

# Show what was written to disk
print("\nFirst 3 lines from cnn_results.jsonl:")
with open("cnn_results.jsonl") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        data = json.loads(line)
        print(f"  • {data['title'][:50]} ({data['internal_links']} internal links)")

## Callbacks: Stream Results Without Memory Buildup

For large crawls, callbacks allow processing documents as they're crawled without accumulating everything in memory.